# Sprint 11 · Webinar 35 · BI (Power BI + Tableau) · Clase práctico-teórica (100 min)
**Tema:** Modelado de datos, medidas avanzadas y dashboards interactivos

> En esta sesión conectarás múltiples fuentes de datos, construirás un modelo en estrella,
> escribirás medidas DAX (Power BI) y expresiones LOD (Tableau), y diseñarás paneles
> interactivos adaptados a distintos públicos.

**Programa:** Data Analytics · **Sprint:** 11 · **Duración:** 100 min · **Modalidad:** Práctico

> ⚠️ **Importante:** Python se usa **únicamente** para generar los datasets de la sesión.
> Todos los ejercicios de modelado, cálculo y visualización se realizan directamente en
> **Power BI Desktop** y **Tableau Public**.

## Fecha
> Completa aquí la fecha de la sesión.


## Objetivos de la sesión

Al finalizar esta clase, podrás:

1. Conectar múltiples fuentes CSV y construir un **modelo en estrella** en Power BI y Tableau.
2. Crear **columnas calculadas, medidas y tablas** usando DAX en Power BI.
3. Aplicar **CALCULATE** para modificar el contexto de filtro en Power BI.
4. Construir métricas de **inteligencia temporal** (YTD, MoM, YoY) con DAX.
5. Visualizar un **análisis de cohortes** de retención en Power BI.
6. Traducir la lógica de CALCULATE a **expresiones LOD** en Tableau.
7. Replicar análisis de tiempo y cohortes en Tableau.
8. Diseñar **dashboards interactivos** adaptados a diferentes audiencias.


## Agenda sugerida (100 minutos)

| Tiempo | Bloque | Herramienta | Contenido |
|---:|---|:---:|---|
| 0–10 | Setup | 🐍 Python | Generar los 5 datasets CSV |
| 10–25 | 11.1 | Power BI + Tableau | Conexión múltiples fuentes y modelo en estrella |
| 25–50 | 11.2.1–11.2.3 | Power BI | DAX: columnas, medidas, CALCULATE y tiempo |
| 50–65 | 11.2.4 | Power BI | Cohortes y visualización en matriz |
| 65–80 | 11.2.5–11.2.6 | Tableau | LOD expressions + tiempo y cohortes |
| 80–95 | 11.3 | Power BI + Tableau | Paneles interactivos para distintos públicos |
| 95–100 | Cierre | — | Takeaways y próximos pasos |


---

# Setup · Generación de los datasets (10 min) 🐍

**Meta:** producir los 5 archivos CSV que usaremos durante toda la sesión en Power BI y Tableau.

## ¿Por qué múltiples tablas?

En el mundo real los datos nunca llegan en una sola tabla plana. Distintos sistemas producen
distintas fuentes que el analista debe conectar:

| Tabla | Sistema origen | Rol en el modelo |
|---|---|---|
| `ventas.csv` | Sistema transaccional (ERP) | **Tabla de hechos** principal |
| `clientes.csv` | CRM | Dimensión — quién compra |
| `productos.csv` | Catálogo de productos | Dimensión — qué se compra |
| `calendario.csv` | Generado | Dimensión de tiempo (requerida para Time Intelligence) |
| `devoluciones.csv` | Sistema de postventa | Segunda tabla de hechos |

## Esquema del modelo que construiremos

```
         ┌─────────────┐
         │  calendario │
         └──────┬──────┘
           fecha│
┌──────────┐    │fecha    ┌───────────────┐  producto_id  ┌─────────────┐
│ clientes │────┤    ─────│    ventas     │───────────────│  productos  │
└──────────┘ cliente_id  │ (tabla hechos)│               └─────────────┘
                         └───────┬───────┘
                            order_id│
                         ┌─────────┴──────┐
                         │  devoluciones  │
                         └────────────────┘
```

> Esta arquitectura se llama **modelo en estrella** (_star schema_): una tabla de hechos
> central rodeada de dimensiones. Es el estándar de facto en BI.

### Instrucciones
1. Ejecuta la celda de **Imports** primero.
2. Ejecuta la celda **Generador de datos** — creará la carpeta `dataset_sprint11_w35/` con los 5 CSVs.
3. Verifica el output: debes ver los conteos de filas de cada tabla.
4. A partir del siguiente ejercicio **no volvemos a Python** — todo es en Power BI y Tableau.


### Imports y configuración

In [ ]:
# ============================================================
# Imports y configuración — ejecuta esta celda primero
# ============================================================

import numpy as np
import pandas as pd
import os

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

OUTPUT_DIR = 'dataset_sprint11_w35'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Librerías cargadas.')
print('📁 Carpeta de salida:', OUTPUT_DIR)


### Generador de datos

In [ ]:
# ============================================================
# GENERADOR DE DATASETS — Webinar 35 Sprint 11
# Ejecuta este bloque completo para obtener los 5 CSV
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

# ── 1. DIMENSIÓN: CLIENTES ───────────────────────────────────
N_CLIENTES = 500
ciudades  = ['Bogotá','Medellín','Cali','Barranquilla','Cartagena','Bucaramanga','Pereira']
segmentos = ['Consumidor','Corporativo','PYME']
generos   = ['M','F','No especificado']

clientes = pd.DataFrame({
    'cliente_id'    : [f'CLI{str(i).zfill(4)}' for i in range(1, N_CLIENTES+1)],
    'nombre'        : [f'Cliente_{i}' for i in range(1, N_CLIENTES+1)],
    'ciudad'        : rng.choice(ciudades, N_CLIENTES, p=[.30,.22,.18,.12,.08,.05,.05]),
    'segmento'      : rng.choice(segmentos, N_CLIENTES, p=[.55,.25,.20]),
    'genero'        : rng.choice(generos, N_CLIENTES, p=[.45,.45,.10]),
    'fecha_registro': pd.to_datetime(
        rng.integers(
            pd.Timestamp('2021-01-01').value,
            pd.Timestamp('2023-06-30').value, N_CLIENTES)
    ).normalize(),
    'edad': rng.integers(18, 65, N_CLIENTES),
})

# ── 2. DIMENSIÓN: PRODUCTOS ──────────────────────────────────
N_PRODUCTOS = 80
categorias = {
    'Electrónica': ['Smartphones','Laptops','Accesorios','Audio'],
    'Hogar'      : ['Muebles','Decoración','Cocina'],
    'Ropa'       : ['Ropa Hombre','Ropa Mujer','Calzado'],
    'Deportes'   : ['Fitness','Outdoor','Ciclismo'],
}
cat_list, sub_list = [], []
for cat, subs in categorias.items():
    n = N_PRODUCTOS // len(categorias)
    for sub in subs:
        k = n // len(subs)
        cat_list.extend([cat]*k); sub_list.extend([sub]*k)
while len(cat_list) < N_PRODUCTOS:
    cat_list.append('Electrónica'); sub_list.append('Accesorios')

precio_lista = np.round(rng.uniform(15, 600, N_PRODUCTOS), 2)
margen_ratio = rng.uniform(0.35, 0.60, N_PRODUCTOS)

productos = pd.DataFrame({
    'producto_id'    : [f'PRD{str(i).zfill(3)}' for i in range(1, N_PRODUCTOS+1)],
    'nombre_producto': [f'{sub_list[i-1]}_{i}' for i in range(1, N_PRODUCTOS+1)],
    'categoria'      : cat_list[:N_PRODUCTOS],
    'subcategoria'   : sub_list[:N_PRODUCTOS],
    'precio_lista'   : precio_lista,
    'costo_unitario' : np.round(precio_lista * (1 - margen_ratio), 2),
})

# ── 3. DIMENSIÓN: CALENDARIO ─────────────────────────────────
fechas = pd.date_range('2022-01-01', '2024-03-31', freq='D')
calendario = pd.DataFrame({
    'fecha'        : fechas,
    'año'          : fechas.year,
    'trimestre'    : fechas.quarter,
    'mes_num'      : fechas.month,
    'mes_nombre'   : fechas.month_name(),
    'semana'       : fechas.isocalendar().week.astype(int),
    'dia_semana'   : fechas.day_name(),
    'es_fin_semana': (fechas.dayofweek >= 5).astype(int),
    'año_mes'      : fechas.to_period('M').astype(str),
    'año_trimestre': fechas.year.astype(str) + '-Q' + fechas.quarter.astype(str),
})

# ── 4. TABLA DE HECHOS: VENTAS ───────────────────────────────
N_VENTAS = 12_000
canales  = ['Online','Tienda Física','App Móvil','Marketplace']

# Fechas con estacionalidad: Q4 y mitad de año tienen más ventas
fecha_vals = pd.date_range('2022-01-01', '2024-03-31', freq='D')
pesos = np.where(fecha_vals.month.isin([11,12]), 2.5,
         np.where(fecha_vals.month.isin([6,7]), 1.4, 1.0))
pesos = pesos / pesos.sum()

fechas_venta = rng.choice(fecha_vals, size=N_VENTAS, p=pesos)
cli_idx  = rng.choice(clientes['cliente_id'], N_VENTAS)
prod_idx = rng.choice(productos['producto_id'], N_VENTAS,
                       p=rng.dirichlet(np.ones(N_PRODUCTOS)))
precios_ref = dict(zip(productos['producto_id'], productos['precio_lista']))
precio_real = np.array([precios_ref[p] for p in prod_idx])

ventas = pd.DataFrame({
    'order_id'       : [f'ORD{str(i).zfill(6)}' for i in range(1, N_VENTAS+1)],
    'cliente_id'     : cli_idx,
    'producto_id'    : prod_idx,
    'fecha'          : pd.to_datetime(fechas_venta),
    'cantidad'       : rng.integers(1, 6, N_VENTAS),
    'precio_unitario': np.round(precio_real, 2),
    'descuento_pct'  : np.round(
        rng.choice([0,.05,.10,.15,.20], N_VENTAS, p=[.50,.20,.15,.10,.05]), 2),
    'canal'          : rng.choice(canales, N_VENTAS, p=[.45,.25,.20,.10]),
})
ventas['ingreso_neto'] = np.round(
    ventas['cantidad'] * ventas['precio_unitario'] * (1 - ventas['descuento_pct']), 2
)

# ── 5. DEVOLUCIONES ──────────────────────────────────────────
idx_dev = rng.choice(N_VENTAS, size=int(N_VENTAS * 0.06), replace=False)
motivos = ['Defecto de fábrica','No era lo esperado','Llegó tarde',
           'Talla incorrecta','Duplicado']

devoluciones = pd.DataFrame({
    'devolucion_id'   : [f'DEV{str(i).zfill(5)}' for i in range(1, len(idx_dev)+1)],
    'order_id'        : ventas.iloc[idx_dev]['order_id'].values,
    'fecha_devolucion': (
        pd.to_datetime(ventas.iloc[idx_dev]['fecha'].values)
        + pd.to_timedelta(rng.integers(2, 30, len(idx_dev)), unit='D')
    ),
    'motivo'          : rng.choice(motivos, len(idx_dev), p=[.30,.28,.18,.14,.10]),
    'cantidad_devuelta': rng.integers(1, 4, len(idx_dev)),
})

# ── Guardar CSVs ─────────────────────────────────────────────
clientes.to_csv(f'{OUTPUT_DIR}/clientes.csv',        index=False)
productos.to_csv(f'{OUTPUT_DIR}/productos.csv',      index=False)
calendario.to_csv(f'{OUTPUT_DIR}/calendario.csv',    index=False)
ventas.to_csv(f'{OUTPUT_DIR}/ventas.csv',            index=False)
devoluciones.to_csv(f'{OUTPUT_DIR}/devoluciones.csv',index=False)

print('✅ Datasets generados exitosamente en:', OUTPUT_DIR)
print()
print(f'  📄 clientes.csv      → {len(clientes):>6,} filas | columnas: {list(clientes.columns)}')
print(f'  📄 productos.csv     → {len(productos):>6,} filas | columnas: {list(productos.columns)}')
print(f'  📄 calendario.csv    → {len(calendario):>6,} filas | columnas: {list(calendario.columns)}')
print(f'  📄 ventas.csv        → {len(ventas):>6,} filas | columnas: {list(ventas.columns)}')
print(f'  📄 devoluciones.csv  → {len(devoluciones):>6,} filas | columnas: {list(devoluciones.columns)}')
print()
print('→ Abre Power BI Desktop o Tableau Public y carga estos archivos para continuar.')


### Vista previa de los datasets

Ejecuta esta celda para revisar la estructura de cada tabla antes de ir a las herramientas BI.


In [ ]:
# Vista previa — últimas celdas Python de la sesión
print('=== ventas.csv — primeras 5 filas ===')
display(ventas.head())

print('\n=== clientes.csv — primeras 5 filas ===')
display(clientes.head())

print('\n=== productos.csv — primeras 5 filas ===')
display(productos.head())

print('\n=== calendario.csv — primeras 5 filas ===')
display(calendario.head())

print('\n=== devoluciones.csv — primeras 5 filas ===')
display(devoluciones.head())


---

# Ejercicio 1 · 11.1 Conexión de múltiples fuentes y modelado de relaciones (15 min)

**Meta:** importar las 5 tablas en Power BI y Tableau, configurar las relaciones y verificar
que el modelo filtra correctamente.


## 11.1.1 · Fundamentos del modelado

### Conceptos clave

| Término | Definición |
|---|---|
| **Tabla de hechos** | Registra eventos o transacciones. Cada fila = algo que ocurrió. |
| **Tabla de dimensiones** | Describe el contexto de los hechos: quién, qué, cuándo, dónde. |
| **Llave primaria (PK)** | Identificador único en una dimensión. Ej: `cliente_id` en `clientes`. |
| **Llave foránea (FK)** | Referencia a la PK dentro de la tabla de hechos. |
| **Cardinalidad** | Cuántas filas de una tabla corresponden a filas de otra. |
| **Dirección del filtro** | Desde qué tabla fluye el filtro hacia las demás. |

### Modelo en estrella (star schema)

```
    [clientes]     [calendario]    [productos]
         \               |               /
          \              |              /
          [       ventas (hechos)      ]
                         |
                  [devoluciones]
```

Las dimensiones describen los hechos. Los filtros fluyen **desde la dimensión hacia los hechos**
(nunca al revés, salvo que lo configures explícitamente).

### Cardinalidades del modelo

| Relación | Cardinalidad | Tipo |
|---|---|---|
| `clientes` → `ventas` | 1 cliente puede tener muchas ventas | 1 : N |
| `productos` → `ventas` | 1 producto puede aparecer en muchas ventas | 1 : N |
| `calendario` → `ventas` | 1 fecha puede tener muchas ventas | 1 : N |
| `ventas` → `devoluciones` | 1 orden puede tener 0 o 1 devolución | 1 : 0..1 |


## 11.1.2 · Relaciones y comportamiento del modelo en Power BI

### Paso 1 — Importar los 5 CSV

1. Abre **Power BI Desktop**.
2. Cinta `Inicio` → `Obtener datos` → `Texto/CSV`.
3. Navega a la carpeta `dataset_sprint11_w35/` y selecciona `ventas.csv` → `Cargar`.
4. Repite para `clientes.csv`, `productos.csv`, `calendario.csv` y `devoluciones.csv`.
5. En el panel **Datos** (derecha) deberías ver las 5 tablas listadas.

---

### Paso 2 — Ir a la Vista de modelo

1. Haz clic en el ícono de diagrama en el panel izquierdo (**Vista de modelo**).
2. Reorganiza las tablas visualmente:
   - `ventas` al centro.
   - `clientes`, `productos`, `calendario` alrededor (como puntas de estrella).
   - `devoluciones` debajo de `ventas`.

---

### Paso 3 — Crear las relaciones

Para cada relación: **arrastra** la columna de la tabla de hechos hasta la columna
correspondiente en la dimensión. Power BI abrirá un diálogo — verifica los parámetros.

| Desde (hechos) | Columna FK | Hacia (dimensión) | Columna PK | Cardinalidad | Filtro |
|---|---|---|---|---|---|
| ventas | `cliente_id` | clientes | `cliente_id` | Varios a uno (\*:1) | Single → |
| ventas | `producto_id` | productos | `producto_id` | Varios a uno (\*:1) | Single → |
| ventas | `fecha` | calendario | `fecha` | Varios a uno (\*:1) | Single → |
| devoluciones | `order_id` | ventas | `order_id` | Varios a uno (\*:1) | Single → |

> ⚠️ **No actives** el filtro bidireccional (Cross filter) a menos que lo necesites explícitamente.
> Puede causar ambigüedad y resultados incorrectos en DAX.

---

### Paso 4 — Marcar la tabla Calendario como tabla de fechas

1. Haz clic sobre la tabla `calendario` en la Vista de modelo.
2. Cinta `Herramientas de tabla` → `Marcar como tabla de fechas`.
3. En el diálogo selecciona la columna `fecha` → **Aceptar**.

> ✅ Este paso es **obligatorio** para que las funciones de Time Intelligence de DAX funcionen
> correctamente (TOTALYTD, SAMEPERIODLASTYEAR, etc.).

---

### Paso 5 — Verificar el modelo

Crea una visualización rápida de prueba:
1. Ve a la **Vista de informe**.
2. Inserta una **Tabla** (panel Visualizaciones).
3. Arrastra `clientes[segmento]` y `ventas[ingreso_neto]` a los valores.
4. Si el modelo está bien configurado, verás los ingresos correctamente agrupados por segmento.
5. Elimina este visual de prueba antes de continuar.

```
Resultado esperado:
  Segmento        Ingreso Neto
  Consumidor      $X,XXX,XXX
  Corporativo     $X,XXX,XXX
  PYME            $X,XXX,XXX
```


## 11.1.3 · Aplicando el modelado de datos en Tableau

### Diferencia conceptual: Tableau usa relaciones lógicas

| Concepto | Power BI | Tableau |
|---|---|---|
| Cómo conecta tablas | Modelo en memoria (VertiPaq) con relaciones físicas | Relaciones lógicas evaluadas en tiempo de consulta |
| Join por defecto | Se define al crear la relación en el modelo | Tableau infiere el tipo de join según el contexto de uso |
| Rendimiento | Más rápido en dashboards con millones de filas | Más flexible para exploración ad-hoc |

---

### Paso 1 — Conectar la primera tabla

1. Abre **Tableau Public** (o Desktop).
2. Panel izquierdo → `Conectar` → `Archivo de texto` → selecciona `ventas.csv`.
3. La tabla `ventas` aparecerá en el lienzo del origen de datos.

---

### Paso 2 — Agregar las demás tablas (relaciones lógicas)

1. En el panel izquierdo (Archivos) arrastra `clientes.csv` al lienzo — aparecerá junto a `ventas`.
2. Tableau mostrará una línea de relación y pedirá las columnas de coincidencia:
   - `ventas[cliente_id]` = `clientes[cliente_id]` → **Aceptar**.
3. Repite para `productos.csv` (unir por `producto_id`), `calendario.csv` (unir por `fecha`)
   y `devoluciones.csv` (unir por `order_id`).

---

### Paso 3 — Configurar cada relación

Haz clic en cada línea de relación para ajustar:

| Relación | Cardinalidad | Integridad referencial |
|---|---|---|
| ventas ↔ clientes | Muchos a uno | Algunos registros coinciden |
| ventas ↔ productos | Muchos a uno | Algunos registros coinciden |
| ventas ↔ calendario | Muchos a uno | Todos los registros coinciden |
| ventas ↔ devoluciones | Uno a muchos | Algunos registros coinciden |

---

### Paso 4 — Verificar el modelo

1. Ve a una **Hoja de trabajo** (pestaña inferior).
2. Arrastra `Segmento` (de clientes) a **Filas**.
3. Arrastra `Ingreso Neto` (de ventas) a **Columnas** — aparecerá como `SUM(Ingreso Neto)`.
4. Si ves ingresos agrupados por segmento sin errores, el modelo está bien configurado.

> 💡 **Tip Tableau:** a diferencia de Power BI, Tableau crea jerarquías de fecha automáticamente
> (Año → Trimestre → Mes → Día) sin necesidad de una tabla calendario separada.
> Sin embargo, la tabla `calendario.csv` nos da columnas adicionales como `año_trimestre`
> y `es_fin_semana` que usaremos más adelante.


In [ ]:
# (Sin código — este ejercicio se realiza en Power BI y Tableau)

---

# Ejercicio 2 · 11.2.1 Creación de cálculos en DAX: columnas, medidas y tablas (12 min)

**Meta:** entender la diferencia entre los tres tipos de objetos DAX y crear las primeras
medidas del modelo en Power BI.


## Fundamentos teóricos

**DAX** (_Data Analysis Expressions_) es el lenguaje de cálculo de Power BI.
Permite crear tres tipos de objetos:

| Tipo | Dónde vive | Cuándo se evalúa | Uso principal |
|---|---|---|---|
| **Columna calculada** | Dentro de una tabla | Al refrescar datos | Clasificaciones fijas, concatenaciones, flags |
| **Medida** | En cualquier tabla (o carpeta) | Al renderizar cada visual | KPIs, totales, ratios dinámicos |
| **Tabla calculada** | Nueva tabla en el modelo | Al refrescar datos | Calendarios, tablas de parámetros |

> 🧠 **Regla de oro:** prefiere siempre **medidas** sobre columnas calculadas.
> Las medidas son dinámicas — respetan los filtros del visual en tiempo real.
> Las columnas se calculan una sola vez y no cambian con los filtros.

### Sintaxis básica

```dax
-- Medida simple
Total Ventas = SUM(ventas[ingreso_neto])

-- Medida con división segura (evita error de división por cero)
Ticket Promedio = DIVIDE(SUM(ventas[ingreso_neto]), DISTINCTCOUNT(ventas[order_id]))

-- Columna calculada (requiere RELATED para acceder a otra tabla)
ventas[Margen_USD] = ventas[ingreso_neto] - RELATED(productos[costo_unitario]) * ventas[cantidad]
```

---

## Pasos en Power BI Desktop

### A) Crear columnas calculadas en la tabla ventas

1. En el panel **Datos**, selecciona la tabla `ventas`.
2. Cinta `Herramientas de tabla` → **Nueva columna**.
3. Escribe cada fórmula en la barra de fórmulas y presiona Enter:

```dax
-- Columna 1: margen en USD (usa RELATED para acceder a productos)
Margen_USD = ventas[ingreso_neto] - RELATED(productos[costo_unitario]) * ventas[cantidad]

-- Columna 2: label del trimestre como texto
Trimestre_Texto = "Q" & QUARTER(ventas[fecha])

-- Columna 3: flag si tiene descuento
Tiene_Descuento = IF(ventas[descuento_pct] > 0, "Con descuento", "Sin descuento")
```

4. Verifica en la Vista de tabla que las nuevas columnas aparecen correctamente.

---

### B) Crear medidas base

1. Haz clic derecho sobre la tabla `ventas` en el panel Datos → **Nueva medida**.
2. Crea cada medida. Organízalas en una carpeta: en la barra de fórmulas, antes del nombre
   escribe `_Medidas Base` en la propiedad **Carpeta de presentación** del panel Propiedades.

```dax
Total Ventas = SUM(ventas[ingreso_neto])

Nro Órdenes = DISTINCTCOUNT(ventas[order_id])

Ticket Promedio = DIVIDE([Total Ventas], [Nro Órdenes])

Clientes Únicos = DISTINCTCOUNT(ventas[cliente_id])

Tasa Devolución = DIVIDE(COUNTROWS(devoluciones), [Nro Órdenes])

Margen Total = SUM(ventas[Margen_USD])

Margen Pct = DIVIDE([Margen Total], [Total Ventas])
```

---

### C) Verificar en un visual

1. Ve a la Vista de informe → inserta **4 Tarjetas** (panel Visualizaciones).
2. Asigna a cada tarjeta: `Total Ventas`, `Nro Órdenes`, `Ticket Promedio`, `Margen Pct`.
3. Inserta una **Tabla** con `clientes[segmento]` y las medidas `Total Ventas` + `Nro Órdenes`.
4. ¿Los valores cambian cuando filtras por segmento? → las medidas funcionan correctamente. ✅


In [ ]:
# (Sin código — este ejercicio se realiza en Power BI Desktop)

---

# Ejercicio 3 · 11.2.2 Conociendo CALCULATE: el motor de DAX (12 min)

**Meta:** comprender cómo CALCULATE modifica el contexto de filtro y escribir
medidas que responden preguntas condicionales.


## Fundamentos teóricos

**CALCULATE** es la función más poderosa de DAX. Prácticamente cualquier medida
avanzada la utiliza.

```dax
CALCULATE(<expresión>, <filtro1>, <filtro2>, ...)
```

### Lo que hace CALCULATE

1. **Evalúa** la expresión.
2. **Modifica el contexto de filtro** aplicando los filtros que le pasas.
3. El nuevo contexto puede **agregar**, **reemplazar** o **eliminar** filtros existentes.

### El contexto: la idea más importante de DAX

```
SIN CALCULATE:
  Total Ventas en una tarjeta     → total global de toda la tabla
  Total Ventas en tabla por canal → total filtrado por cada canal automáticamente
  → el visual define el contexto

CON CALCULATE:
  Ventas Online = CALCULATE([Total Ventas], ventas[canal] = "Online")
  → FUERZA el filtro canal=Online, independientemente del contexto del visual
  → si el visual ya filtra por ciudad, CALCULATE agrega ese filtro encima
```

### Funciones modificadoras de filtro más usadas

| Función | Efecto sobre el contexto |
|---|---|
| `CALCULATE(expr, filtro)` | Agrega o reemplaza un filtro |
| `ALL(tabla)` | Elimina **todos** los filtros de esa tabla |
| `ALL(tabla[col])` | Elimina el filtro de una columna específica |
| `ALLEXCEPT(tabla, col1, col2)` | Elimina todos los filtros **excepto** los indicados |
| `REMOVEFILTERS(col)` | Alias moderno de ALL para una columna |

---

## Pasos en Power BI Desktop

### A) Medidas por canal

Crea las siguientes medidas (carpeta `_Medidas CALCULATE`):

```dax
Ventas Online = CALCULATE([Total Ventas], ventas[canal] = "Online")

Ventas App = CALCULATE([Total Ventas], ventas[canal] = "App Móvil")

Ventas Tienda = CALCULATE([Total Ventas], ventas[canal] = "Tienda Física")
```

---

### B) Porcentaje del total (uso de ALL)

```dax
-- ALL elimina el contexto de filtro del visual para calcular el total global
% del Total =
    DIVIDE(
        [Total Ventas],
        CALCULATE([Total Ventas], ALL(ventas)),
        0
    )
```

Pruébalo en una **Tabla** con `productos[categoria]` + `Total Ventas` + `% del Total`.
Cada fila debe mostrar el porcentaje de esa categoría sobre el total global.

---

### C) Medidas con filtros cruzados de dimensiones

```dax
-- Ventas del segmento Corporativo (filtro sobre dimensión externa)
Ventas Corporativo =
    CALCULATE([Total Ventas], clientes[segmento] = "Corporativo")

-- Ventas con descuento alto
Ventas Descuento Alto =
    CALCULATE([Total Ventas], ventas[descuento_pct] > 0.10)

-- Ticket promedio solo en canal Online
Ticket Online =
    CALCULATE([Ticket Promedio], ventas[canal] = "Online")
```

---

### D) Verificar con un visual combinado

1. Inserta un **Gráfico de barras agrupadas**.
2. Eje X: `productos[categoria]`.
3. Valores: `Total Ventas` y `Ventas Online` (ambas medidas en el mismo gráfico).
4. Observa que `Ventas Online` siempre es menor que `Total Ventas` — eso confirma
   que CALCULATE está restringiendo correctamente el contexto.

> 💡 **Pregunta para discutir:** ¿qué pasa si en un segmentador filtras por canal = "Online"
> mientras el visual muestra `Ventas Online`? ¿Cambia el resultado? ¿Por qué?


In [ ]:
# (Sin código — este ejercicio se realiza en Power BI Desktop)

---

# Ejercicio 4 · 11.2.3 Analizando datos en el tiempo con DAX (12 min)

**Meta:** construir métricas de inteligencia temporal: acumulado anual (YTD),
variación año vs. año (YoY) y variación mes vs. mes (MoM).


## Fundamentos teóricos

Las funciones de **Time Intelligence** en DAX calculan métricas temporales automáticamente.
Requieren:

1. Una tabla de fechas **completa** (sin días faltantes).
2. Que esté **marcada como tabla de fechas** (lo hicimos en el Ejercicio 1).
3. Que tenga una **relación activa** con la tabla de hechos.

### Funciones más usadas

| Función | Qué calcula | Ejemplo de uso |
|---|---|---|
| `TOTALYTD(expr, fechas)` | Acumulado desde el 1 de enero del año en contexto | KPI anual corriente |
| `TOTALMTD(expr, fechas)` | Acumulado desde el 1 del mes en contexto | KPI mensual corriente |
| `SAMEPERIODLASTYEAR(fechas)` | Mismo período del año anterior | Comparativa YoY |
| `DATEADD(fechas, -1, MONTH)` | Período desplazado N unidades | MoM, QoQ |

---

## Pasos en Power BI Desktop

Crea las siguientes medidas (carpeta `_Time Intelligence`).

### A) Acumulados

```dax
-- Ventas acumuladas desde el 1 de enero del año en contexto
Ventas YTD = TOTALYTD([Total Ventas], calendario[fecha])

-- Órdenes acumuladas en el mes corriente
Órdenes MTD = TOTALMTD([Nro Órdenes], calendario[fecha])
```

---

### B) Comparativa con año anterior (YoY)

```dax
-- Ventas del mismo período del año anterior
Ventas LY =
    CALCULATE(
        [Total Ventas],
        SAMEPERIODLASTYEAR(calendario[fecha])
    )

-- Diferencia absoluta vs. año anterior
Var YoY Abs = [Total Ventas] - [Ventas LY]

-- Variación porcentual año vs. año
Var YoY % = DIVIDE([Total Ventas] - [Ventas LY], [Ventas LY], 0)
```

---

### C) Comparativa con mes anterior (MoM)

```dax
-- Ventas del mes anterior
Ventas Mes Anterior =
    CALCULATE(
        [Total Ventas],
        DATEADD(calendario[fecha], -1, MONTH)
    )

-- Variación porcentual mes vs. mes
Var MoM % = DIVIDE([Total Ventas] - [Ventas Mes Anterior], [Ventas Mes Anterior], 0)
```

---

### D) Construir el visual de tendencia temporal

1. Inserta un **Gráfico de líneas**.
2. Eje X: `calendario[año_mes]` (ordenado ascendente).
3. Valores: `Total Ventas` (línea sólida azul) y `Ventas LY` (línea punteada gris).
4. Agrega una **Segunda leyenda** (segundo eje Y) para `Var YoY %` — cámbialo a porcentaje en el formato.
5. Inserta una **Tabla** debajo con columnas: `año_mes` / `Total Ventas` / `Ventas LY` / `Var YoY %` / `Var MoM %`.

---

### E) Crear tarjetas dinámicas con indicadores

1. Inserta una **Tarjeta** para `Ventas YTD`.
2. Inserta otra para `Var YoY %` — en el panel Formato activa **Formato condicional** → color rojo si < 0, verde si > 0.
3. Agrega un **Segmentador** de `calendario[año]` para que el usuario pueda cambiar el año de análisis.


In [ ]:
# (Sin código — este ejercicio se realiza en Power BI Desktop)

---

# Ejercicio 5 · 11.2.4 Analizando y visualizando cohortes en Power BI (13 min)

**Meta:** construir un análisis de retención por cohorte de clientes y visualizarlo
como un mapa de calor (heatmap) en Power BI.


## Fundamentos teóricos

### ¿Qué es un análisis de cohortes?

Una **cohorte** es un grupo de clientes que comparten el mismo evento de inicio — en este caso,
su primera compra. El análisis de cohortes rastrea el comportamiento de ese grupo a lo largo del tiempo.

Responde preguntas clave:
- ¿Qué porcentaje de clientes adquiridos en enero 2022 volvieron a comprar en febrero?
- ¿Mejora la retención con el tiempo (señal de product-market fit)?
- ¿Cuál fue la cohorte más valiosa en términos de LTV (Lifetime Value)?

### La estructura de un análisis de cohortes

```
         │ M+0  │ M+1  │ M+2  │ M+3  │ M+6  │ M+12 │
─────────┼──────┼──────┼──────┼──────┼──────┼──────┤
Ene-2022 │ 100% │  42% │  28% │  21% │  15% │   9% │  ← % que volvió a comprar
Feb-2022 │ 100% │  39% │  25% │  19% │  13% │   8% │
Mar-2022 │ 100% │  41% │  27% │  22% │  16% │  10% │
```

- **Fila** = cohorte (mes de primera compra)
- **Columna** = mes relativo (0 = mes de adquisición, 1 = siguiente mes, etc.)
- **Valor** = % de clientes de esa cohorte que compraron en ese mes relativo

---

## Pasos en Power BI Desktop

### A) Crear columnas calculadas para la cohorte

Las columnas calculadas son necesarias aquí porque la primera compra del cliente es un
atributo fijo — no cambia con los filtros del visual.

Selecciona la tabla `ventas` → `Nueva columna` y crea:

```dax
-- Columna 1: fecha de primera compra de cada cliente
-- ALLEXCEPT mantiene solo el filtro de cliente_id (ignora fecha, canal, etc.)
Primera_Compra =
    CALCULATE(
        MIN(ventas[fecha]),
        ALLEXCEPT(ventas, ventas[cliente_id])
    )

-- Columna 2: mes-año de la cohorte (texto para la matriz)
Cohorte = FORMAT(ventas[Primera_Compra], "YYYY-MM")

-- Columna 3: meses transcurridos desde la primera compra
Mes_Relativo = DATEDIFF(ventas[Primera_Compra], ventas[fecha], MONTH)
```

---

### B) Crear medidas de retención

```dax
-- Número de clientes únicos en el contexto actual (cohorte × período)
Clientes Cohorte = DISTINCTCOUNT(ventas[cliente_id])

-- Clientes en el mes 0 de la misma cohorte (denominador de retención)
Clientes Mes 0 =
    CALCULATE(
        [Clientes Cohorte],
        ventas[Mes_Relativo] = 0
    )

-- Tasa de retención: % de clientes del mes 0 que compraron en el mes relativo actual
Retención % = DIVIDE([Clientes Cohorte], [Clientes Mes 0], 0)
```

---

### C) Construir la matriz de cohortes (heatmap)

1. Inserta una visualización **Matriz** (panel Visualizaciones).
2. Configura los campos:
   - **Filas:** `ventas[Cohorte]`
   - **Columnas:** `ventas[Mes_Relativo]`
   - **Valores:** medida `Retención %`
3. En el panel **Formato** → `Valores` → formatea como porcentaje.

---

### D) Aplicar formato condicional (heatmap)

1. En el panel Formato de la Matriz → `Formato condicional` → `Color de fondo`.
2. Elige escala de colores:
   - Mínimo: rojo (`#F44336`)
   - Medio: amarillo (`#FFC107`)
   - Máximo: verde (`#4CAF50`)
3. Activa **Mostrar en** → `Solo valores`.

---

### E) Agregar análisis de LTV por cohorte

```dax
-- Ingreso total acumulado por cada cohorte (LTV de la cohorte)
LTV Cohorte = CALCULATE([Total Ventas], ALLEXCEPT(ventas, ventas[Cohorte]))
```

Agrega `LTV Cohorte` como segunda medida en la matriz (en los Valores).
Cambia el tipo de visualización a **Tabla** para verla mejor.

> 💡 **Insight esperado:** las primeras cohortes (2022) tendrán mayor LTV porque llevan
> más meses comprando. La retención en M+1 suele estar entre 35–45% — un benchmark razonable
> para e-commerce.


In [ ]:
# (Sin código — este ejercicio se realiza en Power BI Desktop)

---

# Ejercicio 6 · 11.2.5 Creando cálculos en Tableau: de CALCULATE a LOD (10 min)

**Meta:** entender las expresiones LOD de Tableau y traducir la lógica de CALCULATE
a la sintaxis de Tableau.


## Fundamentos teóricos

En Tableau, el equivalente a `CALCULATE` con `ALL`/`ALLEXCEPT` son las
**expresiones LOD** (_Level of Detail_). Permiten calcular una medida a un nivel
de granularidad **diferente** al del visual actual.

### Las tres formas de LOD

| Tipo | Sintaxis | Qué hace |
|---|---|---|
| **FIXED** | `{ FIXED [dim] : AGG(medida) }` | Ignora el contexto del visual; calcula SIEMPRE al nivel de `[dim]` |
| **INCLUDE** | `{ INCLUDE [dim] : AGG(medida) }` | Agrega una dimensión al nivel del visual (más detallado) |
| **EXCLUDE** | `{ EXCLUDE [dim] : AGG(medida) }` | Quita una dimensión del nivel del visual (menos detallado) |

### Tabla de equivalencias DAX ↔ LOD

| Pregunta | DAX (Power BI) | LOD (Tableau) |
|---|---|---|
| Primera compra del cliente | `CALCULATE(MIN(ventas[fecha]), ALLEXCEPT(ventas, ventas[cliente_id]))` | `{ FIXED [cliente_id] : MIN([fecha]) }` |
| % del total global | `DIVIDE([Ventas], CALCULATE([Ventas], ALL(ventas)))` | `SUM([ingreso_neto]) / { FIXED : SUM([ingreso_neto]) }` |
| Ventas promedio por cliente | `DIVIDE([Total Ventas], [Clientes Únicos])` | `AVG({ FIXED [cliente_id] : SUM([ingreso_neto]) })` |
| Total de categoría en fila de subcategoría | `CALCULATE([Ventas], ALLEXCEPT(productos, productos[categoria]))` | `{ FIXED [categoria] : SUM([ingreso_neto]) }` |

> ⚠️ **Regla clave de LOD FIXED:** los campos FIXED **ignoran los filtros de dimensión** del visual,
> pero **sí respetan los filtros de contexto** (la categoría "Filtros de contexto" en el panel de filtros).

---

## Pasos en Tableau

### A) Crear el campo LOD: Primera compra del cliente

1. En el panel de datos, haz clic derecho → **Crear campo calculado**.
2. Nombre: `Primera Compra Cliente`
3. Fórmula:
```
{ FIXED [cliente_id] : MIN([fecha]) }
```
4. Haz clic en **Aceptar** — el campo aparecerá como dimensión (tipo fecha).

---

### B) Crear el campo de Cohorte

1. Nuevo campo calculado → Nombre: `Cohorte`
2. Fórmula:
```
STR(YEAR([Primera Compra Cliente])) + '-'
+ RIGHT('0' + STR(MONTH([Primera Compra Cliente])), 2)
```

---

### C) Porcentaje del total (EXCLUDE equivalente)

1. Nuevo campo calculado → Nombre: `% del Total Ventas`
2. Fórmula:
```
SUM([ingreso_neto]) / { FIXED : SUM([ingreso_neto]) }
```
3. Pruébalo: crea una barra con `categoria` en Filas y `% del Total Ventas` en Columnas.
   La suma de todas las barras debe dar exactamente 100%.

---

### D) LTV promedio por cliente (FIXED + AVG)

1. Nuevo campo calculado → Nombre: `Ventas Totales por Cliente (LOD)`
2. Fórmula:
```
{ FIXED [cliente_id] : SUM([ingreso_neto]) }
```
3. En el visual, usa `AVG([Ventas Totales por Cliente (LOD)])` para obtener el LTV promedio
   aunque el visual esté agrupado por otra dimensión (ej: ciudad o segmento).

---

### E) Verificar con una vista de tabla

1. Arrastra `segmento` a Filas.
2. Arrastra `SUM(ingreso_neto)` y `% del Total Ventas` a Columnas.
3. Compara con los valores que viste en Power BI en el Ejercicio 2.
   Deben coincidir (mismo dataset, misma lógica). ✅


In [ ]:
# (Sin código — este ejercicio se realiza en Tableau)

---

# Ejercicio 7 · 11.2.6 Analizando tiempo y cohortes en Tableau (10 min)

**Meta:** replicar los análisis de inteligencia temporal y cohortes en Tableau
usando cálculos de tabla y expresiones LOD.


## Fundamentos teóricos

Tableau maneja el tiempo de forma diferente a Power BI:

| Aspecto | Power BI (DAX) | Tableau |
|---|---|---|
| Jerarquía de fechas | Manual (tabla calendario + columnas calculadas) | **Automática** al soltar un campo de fecha |
| Acumulado YTD | `TOTALYTD` (función explícita) | Cálculo de tabla → Suma continua |
| Período anterior | `SAMEPERIODLASTYEAR`, `DATEADD` | Cálculo de tabla → % de diferencia, o LOD |
| Filtro de fechas | Segmentador de tipo fecha | Filtro nativo con selector de rango |

---

## Pasos en Tableau — Análisis temporal

### A) Evolución mensual de ventas

1. Arrastra `fecha` al estante de **Columnas** — Tableau crea la jerarquía automáticamente.
2. Haz clic derecho sobre `YEAR(fecha)` en Columnas → selecciona **Mes** (discreto).
   O selecciona `Mes/Año` continuo para una línea sin interrupciones.
3. Arrastra `ingreso_neto` al estante de **Filas** → `SUM(ingreso_neto)`.
4. Cambia el tipo de marca a **Línea** (menú desplegable de marcas).
5. Filtra por `calendario[año]` para ver un año específico.

---

### B) Acumulado YTD (suma continua)

1. Con el gráfico de líneas mensual ya creado:
2. Haz clic derecho sobre el eje de `SUM(ingreso_neto)` → **Agregar cálculo de tabla**.
3. Tipo de cálculo: **Suma continua** → Calcular usando: **Fecha**.
4. La línea se convierte ahora en el acumulado del año (equivalente a `Ventas YTD` de Power BI).
5. Duplica la hoja para mantener la vista mensual original.

---

### C) Variación YoY

1. En el gráfico mensual, haz clic derecho sobre `SUM(ingreso_neto)` → **Agregar cálculo de tabla**.
2. Tipo de cálculo: **Porcentaje de diferencia desde** → Valor relativo: **Año anterior**.
3. Esto muestra la variación % respecto al mismo mes del año anterior (equivalente a `Var YoY %`).
4. Agrega `Etiquetas de marca` para ver los valores en el gráfico.

---

### D) Comparativa de años (dual lines)

1. Nueva hoja → `fecha` en Columnas (nivel: **Mes**, discreto).
2. `ingreso_neto` en Filas.
3. Arrastra `calendario[año]` al estante de **Color** en la tarjeta de Marcas.
4. Ahora tienes una línea por año — equivalente al gráfico de evolución que creaste en Power BI.
5. Cambia el color del año 2024 a naranja o rojo para destacar el año en curso.

---

## Pasos en Tableau — Análisis de cohortes

### E) Crear el campo Mes Relativo

1. Nuevo campo calculado → Nombre: `Mes Relativo`
2. Fórmula:
```
DATEDIFF('month', [Primera Compra Cliente], [fecha])
```

---

### F) Construir la matriz de cohortes

1. Nueva hoja de trabajo.
2. Arrastra `Cohorte` al estante de **Filas**.
3. Arrastra `Mes Relativo` al estante de **Columnas**.
   → Haz clic derecho → **Dimensión** (para que no sume sino que agrupe).
4. Arrastra `cliente_id` a **Texto** en la tarjeta de Marcas → clic derecho → **Medida** → **Recuento (Distintos)**.
5. Ahora tienes el conteo de clientes únicos por cohorte y mes relativo.

---

### G) Calcular la tasa de retención

1. Nuevo campo calculado → Nombre: `Clientes Mes 0 por Cohorte`
2. Fórmula:
```
{ FIXED [Cohorte] : COUNTD(IF [Mes Relativo] = 0 THEN [cliente_id] END) }
```
3. Nuevo campo calculado → Nombre: `Retención %`
4. Fórmula:
```
COUNTD([cliente_id]) / [Clientes Mes 0 por Cohorte]
```
5. Reemplaza el texto de la matriz por `Retención %` → formatea como porcentaje.

---

### H) Aplicar el heatmap de colores

1. Arrastra `Retención %` a **Color** en la tarjeta de Marcas.
2. Haz clic en `Color` → **Editar colores** → Paleta: `Rojo-Verde divergente`.
3. Activa **Usar rango completo** y marca **Centrado** en 0.5 (50%).
4. Haz clic en **Aplicar**.

> 💡 **Tip:** para una vista más limpia, filtra `Mes Relativo` para mostrar solo de 0 a 12,
> y `Cohorte` para mostrar cohortes con al menos 6 meses de historia (excluye las más recientes).


In [ ]:
# (Sin código — este ejercicio se realiza en Tableau)

---

# Ejercicio 8 · 11.3 Diseño de paneles interactivos para diferentes públicos (15 min)

**Meta:** diseñar y construir un dashboard completo en Power BI y uno en Tableau,
adaptados a audiencias distintas con interactividad real.


## Fundamentos teóricos

### ¿Por qué el público define el diseño?

El mismo conjunto de datos puede y debe contar historias distintas según quién lo mira:

| Audiencia | Necesidad principal | Nivel de detalle | Elementos clave |
|---|---|---|---|
| **Ejecutivo (C-Suite)** | Decisión estratégica rápida | Muy alto nivel | 3–4 KPIs, 1 gráfico de tendencia, semáforos |
| **Gerente de área** | Gestionar y ajustar | Nivel medio | Filtros por área, comparativos, alertas |
| **Analista / Operaciones** | Investigar y actuar | Máximo detalle | Drill-through, tablas, cohortes, exportar |

### Principios de diseño universal para dashboards

1. **Una pregunta, un dashboard.** Antes de diseñar, escribe la pregunta que debe responder.
2. **Jerarquía visual:** lo más importante debe ser lo más grande y visible (arriba izquierda).
3. **Relación señal/ruido:** elimina bordes innecesarios, colores de fondo, fuentes decorativas.
4. **Máximo 3 colores:** 1 color principal, 1 acento, 1 color de alerta (rojo = problema).
5. **Consistencia cromática:** el mismo color siempre significa lo mismo en todo el dashboard.
6. **Accesibilidad:** evita rojo/verde juntos (daltonismo). Usa naranja/azul como alternativa.
7. **Acción → reacción clara:** el usuario debe saber qué pasa al hacer clic en cada elemento.

---

## Pasos en Power BI — Dashboard Ejecutivo

### Estructura del panel

```
┌────────────────────────────────────────────────────────────┐
│  [Logo / Título]                    [Fecha: último update] │
├──────────┬──────────┬──────────┬───────────────────────────┤
│ Ventas   │ Órdenes  │  Ticket  │ Margen %    │ Tasa Dev.  │
│  YTD     │  YTD     │ Promedio │             │            │
├──────────┴──────────┴──────────┴─────────────┴────────────┤
│           Evolución mensual: Total Ventas vs. Ventas LY    │
│      ───────────────────────────────────────────────────   │
├────────────────────────┬───────────────────────────────────┤
│  Ventas por categoría  │   % por canal (gráfico de dona)   │
│  (barras horizontales) │                                   │
└────────────────────────┴───────────────────────────────────┘
```

### Paso a paso

1. Crea una **nueva página** → renómbrala `Ejecutivo`.
2. Panel Formato → `Fondo de página` → color `#F5F5F5` (gris muy claro).
3. Inserta **5 Tarjetas** en la fila superior:
   - `Ventas YTD`, `Órdenes MTD`, `Ticket Promedio`, `Margen Pct`, `Tasa Devolución`.
   - Formato: fuente grande (24–28 pt), título pequeño gris, sin borde, fondo blanco.
4. Inserta un **Gráfico de líneas**:
   - Eje X: `calendario[año_mes]`.
   - Valores: `Total Ventas` (azul, línea gruesa) + `Ventas LY` (gris, línea punteada).
   - Activa **Línea de tendencia** (panel Análisis del visual).
5. Inserta un **Gráfico de barras horizontales**:
   - Eje Y: `productos[categoria]` (ordenado por valor desc.).
   - Valor X: `Total Ventas`.
   - Activa **Etiquetas de datos**.
6. Inserta un **Gráfico de dona**:
   - Leyenda: `ventas[canal]`.
   - Valores: `Total Ventas`.
7. Agrega un **Segmentador** de `calendario[año]` (estilo: lista o desplegable).
8. Activa **Interactividad visual:** cinta `Formato` → `Editar interacciones`.
   Configura que al hacer clic en una categoría del gráfico de barras,
   todos los demás visuales se filtren.

---

## Pasos en Power BI — Dashboard Gerencial

1. Nueva página → `Gerencial`.
2. Agrega un **panel de filtros** en la parte superior:
   - Segmentadores: `canal`, `segmento`, `productos[categoria]`, `calendario[año_trimestre]`.
   - Alinea horizontalmente todos los segmentadores.
3. Inserta una **Tabla** con:
   `productos[categoria]` / `Total Ventas` / `Var YoY %` / `Nro Órdenes` / `Tasa Devolución`.
   Aplica formato condicional a `Var YoY %` (rojo si negativo, verde si positivo).
4. Inserta un **Gráfico de dispersión**:
   - Eje X: `Ticket Promedio`.
   - Eje Y: `Nro Órdenes`.
   - Tamaño de burbuja: `Total Ventas`.
   - Leyenda: `productos[categoria]`.
5. Configura **Drill-through**:
   - Nueva página → `Detalle Producto`.
   - En el campo `Drill-through` arrastra `productos[nombre_producto]`.
   - Agrega una tabla con todas las ventas de ese producto.
   - En el visual de categorías del dashboard Gerencial, clic derecho → `Obtener detalles`.

---

## Pasos en Tableau — Dashboard Analítico

### A) Preparar las hojas de trabajo necesarias

Antes de construir el dashboard, asegúrate de tener estas hojas:
1. `Tendencia Mensual` — línea de ventas con años comparados (del Ejercicio 7).
2. `Ventas por Categoría` — barras horizontales ordenadas por valor.
3. `Cohortes Retención` — matriz de heatmap (del Ejercicio 7).
4. `Top 10 Clientes` — tabla con `cliente_id`, `segmento`, total ventas, nro órdenes.

---

### B) Crear el Dashboard en Tableau

1. Clic en el ícono `+` de la barra inferior → **Nuevo Dashboard**.
2. Panel izquierdo → `Tamaño` → **Fijo** → 1366 × 768 px.
3. Arrastra las hojas de trabajo al lienzo en el orden del esquema:
```
┌──────────────────────────────────────────────────────────┐
│  Filtros globales: [Canal] [Segmento] [Rango de fechas]  │
├─────────────────────────┬────────────────────────────────┤
│   Tendencia Mensual     │   Ventas por Categoría         │
│   (60% del ancho)       │   (40% del ancho)              │
├─────────────────────────┴────────────────────────────────┤
│               Cohortes — Mapa de Retención               │
└──────────────────────────────────────────────────────────┘
```

---

### C) Configurar filtros globales

1. En cualquier hoja del dashboard, haz clic derecho sobre un filtro → **Aplicar a hojas**
   → **Todas las hojas que usan esta fuente de datos**.
2. Repite para los filtros de `canal`, `segmento` y `año`.
3. Los filtros ahora controlan **todo el dashboard** — no solo la hoja donde los creaste.

---

### D) Configurar acciones de interactividad

1. Menú `Dashboard` → `Acciones` → `Agregar acción` → **Filtro**.
2. Configura:
   - **Ejecutar acción en:** Seleccionar.
   - **Hojas de origen:** `Ventas por Categoría`.
   - **Hojas de destino:** `Tendencia Mensual` y `Cohortes Retención`.
   - **Al borrar la selección:** Mostrar todos los valores.
3. Prueba: haz clic en una barra de categoría → los otros visuales deben filtrarse.

---

### E) Ajustes finales de diseño

1. Agrega un **Objeto de texto** en la parte superior con el título del dashboard.
2. Panel `Dashboard` → `Fondo` → color `#FAFAFA`.
3. Para cada hoja: clic derecho → `Formato` → elimina líneas de cuadrícula innecesarias.
4. Activa `Dashboard` → `Diseño de dispositivo` → `Teléfono` para crear
   una versión mobile automáticamente.

---

### ✍️ Actividad de reflexión

Antes de cerrar la sesión, responde en el chat:

| Pregunta | Tu respuesta |
|---|---|
| ¿Para qué audiencia es tu dashboard? | |
| ¿Cuál es la pregunta de negocio que responde? | |
| ¿Qué decisión debería tomar quien lo vea? | |
| ¿Qué elemento cambiarías si el público fuera diferente? | |


In [ ]:
# (Sin código — este ejercicio se realiza en Power BI y Tableau)

---

# Cierre y Takeaways (5 min)

## Los 7 aprendizajes de esta sesión

1. 🔗 **El modelo de datos lo es todo.** Un modelo en estrella bien configurado habilita
   medidas DAX correctas y consultas rápidas. Sin el modelo, los números mienten.

2. 🧮 **Columnas vs. medidas:** las columnas son estáticas (se calculan al refrescar),
   las medidas son dinámicas (responden a los filtros del visual). Prefiere siempre medidas.

3. ⚡ **CALCULATE es el núcleo de DAX.** Permite agregar, reemplazar o eliminar filtros
   del contexto. El 80% de las medidas avanzadas lo usan directamente o de forma indirecta.

4. 📅 **Time Intelligence requiere una tabla de fechas marcada.** Sin ese paso,
   `TOTALYTD` y `SAMEPERIODLASTYEAR` no funcionarán correctamente.

5. 👥 **Los cohortes revelan la salud real del negocio.** Los ingresos pueden crecer
   por adquisición mientras la retención cae — sin cohortes, ese problema es invisible.

6. 📐 **LOD FIXED = CALCULATE + ALLEXCEPT.** Misma lógica, distinta sintaxis.
   Entender el concepto te permite migrar entre herramientas sin reaprender desde cero.

7. 🎨 **Un dashboard no es una colección de gráficos.** Es la respuesta visual a una
   pregunta de negocio específica, diseñada para una audiencia concreta.

---

## Checklist final

Antes de cerrar el trabajo de hoy, verifica:

- [ ] 5 CSV importados y modelo en estrella configurado en Power BI
- [ ] 5 tablas relacionadas en Tableau con relaciones lógicas
- [ ] Medidas base creadas: `Total Ventas`, `Nro Órdenes`, `Ticket Promedio`, `Clientes Únicos`, `Tasa Devolución`
- [ ] Medidas CALCULATE: `Ventas Online`, `% del Total`, `Ventas Corporativo`
- [ ] Time Intelligence: `Ventas YTD`, `Ventas LY`, `Var YoY %`, `Var MoM %`
- [ ] Cohortes: columnas `Primera_Compra`, `Cohorte`, `Mes_Relativo` + medidas `Retención %`
- [ ] Matriz de cohortes con heatmap de colores en Power BI
- [ ] LOD FIXED: `Primera Compra Cliente`, `Cohorte`, `% del Total Ventas` en Tableau
- [ ] Matriz de cohortes con heatmap en Tableau
- [ ] Dashboard ejecutivo (Power BI) con interactividad
- [ ] Dashboard analítico (Tableau) con acciones de filtro

---

## Próximos pasos y recursos

| Recurso | Descripción | Enlace |
|---|---|---|
| DAX Guide | Referencia completa de funciones DAX | https://dax.guide |
| SQLBI — contextos en DAX | Artículo fundamental sobre contextos de filtro y fila | https://www.sqlbi.com/articles/row-context-and-filter-context-in-dax/ |
| Tableau LOD Guide | Documentación oficial de expresiones LOD | https://help.tableau.com/current/pro/desktop/en-us/calculations_calculatedfields_lod.htm |
| Power BI Time Intelligence | Lista completa de funciones de tiempo en DAX | https://learn.microsoft.com/en-us/dax/time-intelligence-functions-dax |
| Color Oracle | Simulador de daltonismo para revisar tus dashboards | https://colororacle.org |
| Storytelling with Data | Libro de referencia para diseño de dashboards | https://www.storytellingwithdata.com |
